<a href="https://colab.research.google.com/github/AntonDozhdikov/politpredict/blob/main/MONOGRAPH_CHAPTER4_FINAL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## ФИНАЛЬНЫЙ исследовательский notebook

### «Традиция и технологическая новизна в ценностной структуре российского общества: идентичность, семья и отношение к искусственному интеллекту»

Это финальная версия экспериментального блока IV раздела коллективной монографии.

**Логика запуска:** `Runtime → Run all`.

Ноутбук автоматически выполняет:

1. загрузку трёх открытых микромассивов ВЦИОМ;
2. загрузку и проверку микроданных Росстата ИКТ за 2022–2025 гг.;
3. строгий NLP-анализ только содержательных кодированных открытых ответов;
4. двухуровневую семантическую типологию: детальные кластеры + публикационные макротипы;
5. проверку устойчивости кластеров;
6. анализ совместной встречаемости смысловых категорий ;
7. семантическое сопоставление «ценности ↔ идеальная семья» и «Родина ↔ Россия» с permutation baseline;
8. два независимых ML-контура:
   - **discrimination** — для различения классов и SHAP;
   - **probability/calibration** — для вероятностей и групповых профилей;
9. bootstrap 95% CI, repeated cross-validation и robustness;
10. графики и диагностические графики отдельно;
11. научную интерпретацию с явным разграничением статистической значимости, величины эффекта и причинности;
12. Excel, CSV, модели и итоговые ZIP.

### Исправления относительно предыдущих версий

- `ID`, веса и служебные поля полностью исключены из NLP;
- исправлено SHAP-индексирование: только positional indexing после `reset_index(drop=True)`;
- class-balanced ML больше **не используется для вероятностной интерпретации**;
- probability-модели обучаются без искусственной балансировки классов;
- OOF-профили для публикации основаны на наблюдаемых survey-weighted долях, а не на искажённых balanced probabilities;
- UMAP остаётся диагностикой, а не основным доказательством;
- network-графики заменены на читаемые матрицы ассоциаций;
- для семантической близости показывается не только `p-value`, но и абсолютная разница с permutation baseline;
- для Росстата строятся отдельные временные ряды ключевых цифровых практик;
- все названия переменных в публикационных графиках переводятся на человеческий язык.

## Исследовательская логика и ограничения

Рабочая гипотеза не предполагает единой шкалы «традиционность ↔ инновационность». Проверяется более сложная конфигурация относительно самостоятельных измерений:

- семейные и нормативные ценности;
- образ идеальной семьи;
- смысл Родины;
- ассоциации со словом «Россия»;
- отношение к применению ИИ;
- поддержка прозрачности и маркировки ИИ-контента;
- допустимая автономия ИИ в высокорисковых решениях;
- социально-демографические и цифровые различия этих установок.

### Ограничения

1. Три массива ВЦИОМ — **независимые выборки**, поэтому между ними не создаётся respondent-level связь.
2. NLP работает с кодированными категориями открытых ответов, а не с исходными дословными ответами.
3. Кластеры являются описательной типологией; они не интерпретируются как «объективно существующие группы» без дополнительных оснований.
4. Значимый permutation test при малой абсолютной разнице трактуется как **малый эффект**, а не как сильная содержательная связь.
5. ML оценивает предсказуемость установок. SHAP и feature importance не являются причинными эффектами.
6. Росстат используется как независимый контекст цифровых практик и не объединяется с ВЦИОМ на уровне индивидов.

## методическая схема

### NLP

Для четырёх содержательных доменов используются только разрешённые поля:

- `sem1_1d`, `sem1_2d`, `sem1_3d` — традиционные ценности;
- `sem3_1d`, `sem3_2d`, `sem3_3d` — идеальная семья;
- `rdn1_1d`, `rdn1_2d`, `rdn1_3d` — смысл Родины;
- `rdn2_1d`, `rdn2_2d`, `rdn2_3d` — ассоциации с Россией.

Эмбеддинги: `multilingual-e5-large` с автоматическим fallback на более лёгкую multilingual-модель.

Два уровня кластеризации:

- **fine clusters** — оптимизация silhouette при ограничении минимальной доли;
- **macro clusters** — 3–5 типов для содержательной интерпретации и публикационных графиков.

Устойчивость: повторные KMeans и Adjusted Rand Index.

### ML

Основные задачи:

- `ii1` — широкое / селективное / отрицательное отношение к применению ИИ;
- `ii4` — поддержка / неподдержка маркировки;
- `ii5` — без ИИ / human-in-the-loop / автономное решение.

**Discrimination track**
- Weighted prior;
- balanced spline multinomial logit;
- balanced CatBoost;
- macro-F1 и balanced accuracy.

**Probability track**
- Weighted prior;
- survey-weighted spline multinomial logit;
- survey-weighted CatBoost без class balancing;
- log-loss, multiclass Brier и calibration.

Все оценки — out-of-fold, repeated stratified 5-fold CV × 2 повтора.  
Неопределённость основных метрик — bootstrap 95% CI.

In [ ]:
# A.1 Установка, конфигурация, тихий режим
import os, sys, re, io, json, time, math, shutil, zipfile, hashlib, warnings, subprocess
from pathlib import Path
from urllib.parse import urljoin, unquote

os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
os.environ["TRANSFORMERS_NO_ADVISORY_WARNINGS"] = "1"
warnings.filterwarnings("ignore")

def quiet_pip(packages):
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", *packages],
        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=True
    )

quiet_pip([
    "pandas>=2.2", "numpy", "requests", "beautifulsoup4", "lxml",
    "openpyxl", "xlsxwriter", "pyarrow", "tqdm", "pyreadstat",
    "scikit-learn>=1.5", "scipy", "catboost", "sentence-transformers", "torch", "matplotlib", "umap-learn", "networkx"
])

import numpy as np
import pandas as pd
import requests
import pyreadstat
import matplotlib.pyplot as plt
import logging
logging.getLogger("huggingface_hub").setLevel(logging.ERROR)
logging.getLogger("sentence_transformers").setLevel(logging.ERROR)
from bs4 import BeautifulSoup
from tqdm.auto import tqdm
from IPython.display import display

SEED = 42
ROOT = Path("/content/MONOGRAPH_CHAPTER4_FINAL")
DATA = ROOT / "DATA"
LOADER = ROOT / "00_LOADER"
RESULTS = ROOT / "RESULTS"
FIG = RESULTS / "figures"
MODELS = RESULTS / "models"
PUB = RESULTS / "publication_figures"
DIAG = RESULTS / "diagnostics"
CACHE = ROOT / "CACHE"
for p in [ROOT, DATA, LOADER, RESULTS, FIG, MODELS, PUB, DIAG, CACHE]:
    p.mkdir(parents=True, exist_ok=True)

UA = ("Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
      "AppleWebKit/537.36 (KHTML, like Gecko) "
      "Chrome/153.0.0.0 Safari/537.36")
HEADERS = {"User-Agent": UA, "Accept-Language": "ru-RU,ru;q=0.9,en;q=0.5", "Accept": "*/*"}
SESSION = requests.Session(); SESSION.headers.update(HEADERS)
BAR_FMT = "{desc}: {percentage:3.0f}%|{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}] {postfix}"

def safe_name(x):
    return re.sub(r"[^0-9A-Za-zА-Яа-яЁё._-]+", "_", str(x)).strip("_")[:180]

def sha256(path, chunk=1024*1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            b = f.read(chunk)
            if not b: break
            h.update(b)
    return h.hexdigest()

def flatten_cols(df):
    x=df.copy()
    if isinstance(x.columns,pd.MultiIndex):
        x.columns=[" | ".join(str(v) for v in c if str(v)!="nan").strip(" |") for c in x.columns]
    else:
        x.columns=[str(c) for c in x.columns]
    return x

def number_or_nan(v):
    if pd.isna(v): return np.nan
    s=str(v).replace("\xa0"," ").replace("−","-").strip().replace(",", ".")
    m=re.search(r"-?\d+(?:\.\d+)?",s)
    return float(m.group()) if m else np.nan

def fetch_bytes(url, timeout=60, referer=None):
    headers=dict(HEADERS)
    if referer: headers["Referer"]=referer
    last=None
    for delay in [0,1.0,2.0]:
        if delay: time.sleep(delay)
        try:
            r=SESSION.get(url,headers=headers,timeout=timeout,allow_redirects=True)
            r.raise_for_status()
            return r.content,r.headers,r.url
        except Exception as e:
            last=e
    tmp=Path("/content/_fetch_tmp.bin")
    cmd=["curl","-L","--retry","2","--connect-timeout","15","--max-time",str(timeout),"-A",UA,"-sS","-o",str(tmp),url]
    if referer: cmd[1:1]=["-e",referer]
    cp=subprocess.run(cmd,stdout=subprocess.DEVNULL,stderr=subprocess.DEVNULL)
    if cp.returncode==0 and tmp.exists() and tmp.stat().st_size>0:
        b=tmp.read_bytes(); tmp.unlink(missing_ok=True); return b,{},url
    raise RuntimeError(f"Не удалось получить {url}: {last}")

def sniff_ext(content,headers=None,url=""):
    headers=headers or {}; ct=str(headers.get("content-type","")).lower(); cd=str(headers.get("content-disposition",""))
    m=re.search(r'filename\*?=(?:UTF-8\'\')?"?([^";]+)',cd,re.I)
    if m:
        ext=Path(unquote(m.group(1))).suffix.lower()
        if ext: return ext
    ext=Path(url.split("?")[0]).suffix.lower()
    if ext in {".zip",".sav",".csv",".xlsx",".xls"}: return ext
    if content[:4]==b"PK\x03\x04": return ".zip"
    if content[:4] in {b"$FL2",b"$FL3"}: return ".sav"
    if content[:8]==bytes.fromhex("D0CF11E0A1B11AE1"): return ".xls"
    head=content[:500].lower()
    if b"<html" in head or b"<!doctype html" in head: return ".html"
    if "csv" in ct or b";" in content[:2000] or b"," in content[:2000]: return ".csv"
    return ".bin"

def save_bytes(content,headers,url,stem,folder):
    folder.mkdir(parents=True,exist_ok=True)
    ext=sniff_ext(content,headers,url)
    p=folder/f"{safe_name(stem)}{ext}"; p.write_bytes(content); return p

def read_csv_auto(path):
    last=None
    for enc,sep in [("utf-8-sig",";"),("utf-8-sig",","),("cp1251",";"),("cp1251",","),("utf-8",";"),("utf-8",",")]:
        try:
            df=pd.read_csv(path,encoding=enc,sep=sep,low_memory=False)
            if df.shape[1]>1: return df
        except Exception as e: last=e
    try: return pd.read_csv(path,sep=None,engine="python",encoding_errors="ignore",low_memory=False)
    except Exception: raise last if last else RuntimeError("CSV не читается")

print("Среда готова:",ROOT)

Среда готова: /content/MONOGRAPH_CHAPTER4_FINAL


In [ ]:
import ssl, urllib.request, urllib.error
import urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# A.2 ЕДИНЫЙ ЗАГРУЗЧИК: функции и источники
WCIOM_PAGES={
 "traditional_values":"https://wciom.ru/analytical-reviews/analiticheskii-obzor/tradicionnye-cennosti-ozhidanie-i-realnost",
 "homeland":"https://wciom.ru/analytical-reviews/analiticheskii-obzor/s-chego-nachinaetsja-rodina",
 "ai_ethics":"https://wciom.ru/analytical-reviews/analiticheskii-obzor/ehtika-iskusstvennogo-intellekta-ii",
}
ROSSTAT_PAGE="https://rosstat.gov.ru/opendata/7708234640-IKT-twenty-twenty-five"

def discover_wciom_links(page_url,html):
    soup=BeautifulSoup(html,"lxml"); links=[]
    for a in soup.find_all("a",href=True):
        raw=str(a.get("href","")).strip(); txt=" ".join(a.stripped_strings).lower()
        if not raw or raw.lower().startswith("javascript:"): continue
        href=urljoin(page_url,raw); h=href.lower()
        if any(x in h for x in [".zip",".sav",".csv",".xlsx",".xls"]) or "massiv" in h or "массив" in txt:
            links.append(href)
    for pat in [r'https?://[^"\'\s<>]+?\.(?:zip|sav|csv|xlsx|xls)(?:\?[^"\'\s<>]*)?',r'["\']([^"\']+?\.(?:zip|sav|csv|xlsx|xls)(?:\?[^"\']*)?)["\']']:
        for m in re.findall(pat,html,flags=re.I):
            links.append(urljoin(page_url,m if isinstance(m,str) else m[0]))
    out=[]
    for u in links:
        if u not in out: out.append(u)
    return out

def candidate_data_files(folder):
    return [p for p in folder.rglob("*") if p.is_file() and p.suffix.lower() in {".sav",".csv",".xlsx",".xls"}]

def inspect_microdata(path):
    try:
        if path.suffix.lower()==".sav":
            df,_=pyreadstat.read_sav(path,apply_value_formats=False); return len(df),len(df.columns),"sav"
        if path.suffix.lower()==".csv":
            df=read_csv_auto(path); return len(df),len(df.columns),"csv"
        if path.suffix.lower() in {".xlsx",".xls"}:
            df=pd.read_excel(path); return len(df),len(df.columns),path.suffix.lower().lstrip(".")
    except Exception: pass
    return 0,0,path.suffix.lower().lstrip(".")

def save_published_tables(source,html,out_dir):
    out_dir.mkdir(parents=True,exist_ok=True)
    try: tables=pd.read_html(io.StringIO(html))
    except Exception: tables=[]
    if tables:
        with pd.ExcelWriter(out_dir/f"{source}.xlsx",engine="xlsxwriter") as w:
            for i,t in enumerate(tables[:80]): flatten_cols(t).to_excel(w,f"T{i:02d}",index=False)
    return len(tables)

def load_one_wciom(source,url,pbar):
    base=DATA/"WCIOM"; raw=base/"raw"/source; ext=base/"extracted"/source; pub=base/"published_tables"
    for p in [raw,ext,pub]: p.mkdir(parents=True,exist_ok=True)
    row={"source":source,"provider":"ВЦИОМ","page":url,"page_ok":False,"microdata_ok":False,"tables":0,"data_files":0,"rows":0,"cols_max":0,"note":""}
    pbar.set_postfix_str(f"ВЦИОМ: {source} — страница")
    content,headers,final_url=fetch_bytes(url,timeout=60)
    html=content.decode("utf-8",errors="ignore"); (raw/"page.html").write_text(html,encoding="utf-8")
    row["page_ok"]="<html" in html.lower() or "<!doctype" in html.lower(); row["tables"]=save_published_tables(source,html,pub); pbar.update(1)
    links=discover_wciom_links(final_url,html); errors=[]; usable=[]
    pbar.set_postfix_str(f"ВЦИОМ: {source} — массив")
    for j,href in enumerate(links[:15]):
        try:
            b,h,u=fetch_bytes(href,timeout=90,referer=url); ext0=sniff_ext(b,h,u)
            if ext0 in {".html",".bin"} and len(b)<250000: continue
            p=save_bytes(b,h,u,f"microdata_{j:02d}",raw)
            if p.suffix.lower()==".zip":
                d=ext/p.stem; d.mkdir(parents=True,exist_ok=True)
                with zipfile.ZipFile(p) as z: z.extractall(d)
            elif p.suffix.lower() in {".sav",".csv",".xlsx",".xls"}: shutil.copy2(p,ext/p.name)
            usable=[]
            for fp in candidate_data_files(ext):
                nr,nc,fmt=inspect_microdata(fp)
                if nr>=100 and nc>=2: usable.append((fp,nr,nc,fmt))
            if usable: break
        except Exception as e: errors.append(repr(e)[:160])
    pbar.update(1)
    row["data_files"]=len(usable); row["microdata_ok"]=len(usable)>0; row["rows"]=int(sum(x[1] for x in usable)); row["cols_max"]=int(max([x[2] for x in usable],default=0))
    if not row["microdata_ok"]: row["note"]=f"Микроданные не разрешены; опубликованных таблиц={row['tables']}; errors={'; '.join(errors[-2:])}"
    return row



# ---------- Росстат: CSV-native архитектура 2025 -> 2022 ----------
# Ключевое изменение: широкие микромассивы разных лет НЕ concat'ятся
# и НЕ пересохраняются в общий parquet. Используются официальные CSV напрямую.

import gc

ROSSTAT_ICT_SOURCES = {
    2025: {
        "page": "https://rosstat.gov.ru/opendata/7708234640-IKT-twenty-twenty-five",
        "data": "data-20260227T1302-structure-20260227T1302.csv",
        "structure": "structure-20260227T1302.csv",
    },
    2024: {
        "page": "https://rosstat.gov.ru/opendata/7708234640-IKT-twenty-twenty-four",
        "data": "data-20250228T2002-structure-20250228T2002.zip",
        "structure": "structure-20250228T2002.csv",
    },
    2023: {
        "page": "https://rosstat.gov.ru/opendata/7708234640-IKT-twenty-twenty-three",
        "data": "data-20240410T1504-structure-20240410T1504.zip",
        "structure": "structure-20240410T1504.csv",
    },
    2022: {
        "page": "https://rosstat.gov.ru/opendata/7708234640-IKT-twenty-twenty-two",
        "data": "data-20230525T1005-structure-20230525T1005.zip",
        "structure": "structure-20230525T1005.csv",
    },
}

def _rosstat_page_html(page_url):
    errors=[]
    for verify in (True,False):
        for u in (page_url,page_url+"?print=1"):
            try:
                r=SESSION.get(
                    u,headers=HEADERS,timeout=(8,25),
                    allow_redirects=True,verify=verify
                )
                if r.status_code==200 and len(r.text)>1000:
                    return r.text,r.url
                errors.append(f"{u}: HTTP {r.status_code}")
            except Exception as e:
                errors.append(f"{u}: {type(e).__name__}")
    raise RuntimeError("Страница Росстата недоступна: "+" | ".join(errors[-8:]))

def _rosstat_filename_candidates(html,fixed_name):
    names=[fixed_name]
    for pat in [
        r'data-[A-Za-z0-9_.-]+-structure-[A-Za-z0-9_.-]+\.(?:csv|zip)',
        r'data-[A-Za-z0-9_.-]+\.(?:csv|zip)'
    ]:
        for m in re.findall(pat,html,flags=re.I):
            if m not in names:
                names.append(m)
    return names

def _rosstat_url_variants(page_url,filename):
    bases=[
        page_url,
        page_url.replace("https://rosstat.gov.ru","https://www.rosstat.gov.ru"),
    ]
    out=[]
    for base in bases:
        u=base.rstrip("/")+"/"+filename
        if u not in out:
            out.append(u)
    return out

def _valid_rosstat_payload(content,filename):
    if content is None or len(content)<500:
        return False,"tiny"
    head=content[:800].lstrip().lower()
    if head.startswith(b"<!doctype") or head.startswith(b"<html") or b"<html" in head:
        return False,"html"
    ext=Path(filename).suffix.lower()
    if ext==".zip":
        try:
            if not zipfile.is_zipfile(io.BytesIO(content)):
                return False,"bad_zip"
        except Exception:
            return False,"bad_zip"
    elif ext==".csv" and b"\x00" in content[:5000]:
        return False,"binary_not_csv"
    return True,"ok"

def _rosstat_download(page_url,filename,timeout=35):
    errors=[]
    urls=_rosstat_url_variants(page_url,filename)

    for verify,tag in [(True,"requests"),(False,"requests_noverify")]:
        for u in urls:
            try:
                r=SESSION.get(
                    u,headers={**HEADERS,"Referer":page_url},
                    timeout=(8,timeout),allow_redirects=True,verify=verify
                )
                if r.status_code==200:
                    ok,why=_valid_rosstat_payload(r.content,filename)
                    if ok:
                        return r.content,r.url,tag
                    errors.append(f"{tag} {u}: {why}")
                else:
                    errors.append(f"{tag} {u}: HTTP {r.status_code}")
            except Exception as e:
                errors.append(f"{tag} {u}: {type(e).__name__}")

    for u in urls:
        tmp=Path("/content/_rosstat_tmp.bin")
        tmp.unlink(missing_ok=True)
        cmd=[
            "curl","-k","--http1.1","-L","--retry","2","--retry-delay","1",
            "--connect-timeout","8","--max-time",str(timeout),
            "-A",UA,"-e",page_url,"-sS","-o",str(tmp),u
        ]
        cp=subprocess.run(cmd,stdout=subprocess.DEVNULL,stderr=subprocess.DEVNULL)
        if cp.returncode==0 and tmp.exists():
            b=tmp.read_bytes()
            tmp.unlink(missing_ok=True)
            ok,why=_valid_rosstat_payload(b,filename)
            if ok:
                return b,u,"curl_insecure_http1"
            errors.append(f"curl {u}: {why}")
        tmp.unlink(missing_ok=True)

    ctx=ssl._create_unverified_context()
    for u in urls:
        try:
            req=urllib.request.Request(u,headers={**HEADERS,"Referer":page_url})
            with urllib.request.urlopen(req,timeout=timeout,context=ctx) as resp:
                b=resp.read()
                final=resp.geturl()
            ok,why=_valid_rosstat_payload(b,filename)
            if ok:
                return b,final,"urllib_unverified"
            errors.append(f"urllib {u}: {why}")
        except Exception as e:
            errors.append(f"urllib {u}: {type(e).__name__}")

    raise RuntimeError(" | ".join(errors[-16:]))

def _decode_scalar(v):
    if isinstance(v,(bytes,bytearray,memoryview)):
        b=bytes(v)
        for enc in ("utf-8","cp1251","latin1"):
            try:
                return b.decode(enc)
            except Exception:
                pass
        return b.hex()
    return v

def _coerce_numeric_series(s):
    """
    Безопасно превращает bytes/str/float в numeric.
    Используется в т.ч. для VESA_DX.
    """
    z=s.map(_decode_scalar)
    z=z.astype("string").str.strip().str.replace("\xa0","",regex=False)
    z=z.str.replace(",",".",regex=False)
    return pd.to_numeric(z,errors="coerce")

def _parse_rosstat_resource(content,filename,year,raw_dir,ext_dir):
    raw_dir.mkdir(parents=True,exist_ok=True)
    ext_dir.mkdir(parents=True,exist_ok=True)

    suffix=Path(filename).suffix.lower()
    raw_path=raw_dir/f"rosstat_ict_{year}{suffix}"
    raw_path.write_bytes(content)

    if suffix==".zip":
        target=ext_dir/str(year)
        if target.exists():
            shutil.rmtree(target)
        target.mkdir(parents=True,exist_ok=True)
        with zipfile.ZipFile(raw_path) as z:
            z.extractall(target)
        csv_files=[p for p in target.rglob("*.csv") if p.is_file()]
    else:
        csv_files=[raw_path]

    parsed=[]
    for fp in csv_files:
        try:
            df=read_csv_auto(fp)
            if len(df)>0 and df.shape[1]>1:
                parsed.append((len(df)*df.shape[1],fp,len(df),df.shape[1]))
            del df
        except Exception:
            pass

    if not parsed:
        raise RuntimeError(f"{year}: ресурс получен, но валидный CSV не прочитан")

    parsed.sort(key=lambda x:x[0],reverse=True)
    _,fp,nr,nc=parsed[0]
    return raw_path,fp,int(nr),int(nc)

def _read_structure(content):
    tmp=Path("/content/_rosstat_structure.csv")
    tmp.write_bytes(content)
    try:
        df=read_csv_auto(tmp)
    finally:
        tmp.unlink(missing_ok=True)
    return df

def _structure_mapping(structure_df,data_columns):
    """
    Универсальный словарь: ищем в строке structure.csv значение,
    совпадающее с кодом реальной переменной.
    """
    dcols={str(c) for c in data_columns}
    rows=[]
    if structure_df is None or structure_df.empty:
        return pd.DataFrame(columns=["variable","description"])

    for _,r in structure_df.iterrows():
        vals=[str(v).strip() for v in r.tolist() if pd.notna(v) and str(v).strip()]
        code=next((v for v in vals if v in dcols),None)
        if code:
            desc=" | ".join(v for v in vals if v!=code)
            rows.append({"variable":code,"description":desc})

    return pd.DataFrame(rows).drop_duplicates("variable")

def load_rosstat_ict(pbar):
    base=DATA/"ROSSTAT_ICT"
    raw=base/"raw"
    ext=base/"extracted"
    struct_dir=base/"structure"
    for p in [raw,ext,struct_dir]:
        p.mkdir(parents=True,exist_ok=True)

    row={
        "source":"rosstat_ict_2022_2025","provider":"Росстат",
        "page":ROSSTAT_ICT_SOURCES[2025]["page"],
        "page_ok":False,"microdata_ok":False,"tables":0,
        "data_files":0,"rows":0,"cols_max":0,"note":""
    }

    # 1/2: страницы + актуальные имена
    pbar.set_postfix_str("Росстат ИКТ — страницы / имена")
    resolved={}
    for year,spec in ROSSTAT_ICT_SOURCES.items():
        try:
            html,final=_rosstat_page_html(spec["page"])
            (raw/f"page_{year}.html").write_text(html,encoding="utf-8")
            names=_rosstat_filename_candidates(html,spec["data"])
        except Exception:
            names=[spec["data"]]
        resolved[year]={
            "page":spec["page"],
            "data_names":names,
            "structure":spec["structure"]
        }
    row["page_ok"]=True
    pbar.update(1)

    # 2/2: данные — каждый год автономно
    pbar.set_postfix_str("Росстат ИКТ — CSV 2025→2022")
    year_rows=[]
    schema_parts=[]
    total_rows=0
    max_cols=0

    for year in sorted(resolved,reverse=True):
        info=resolved[year]
        ok=False
        err=""
        transport=""
        used_url=""
        used_name=""
        csv_path=""
        structure_ok=False
        nr=0
        nc=0

        for filename in info["data_names"][:5]:
            try:
                content,used_url,transport=_rosstat_download(
                    info["page"],filename,timeout=35
                )
                raw_path,parsed_path,nr,nc=_parse_rosstat_resource(
                    content,filename,year,raw,ext
                )

                # ВАЖНО: CSV реально перечитывается перед подтверждением статуса.
                test=read_csv_auto(parsed_path)
                if len(test)!=nr or test.shape[1]!=nc:
                    raise RuntimeError("Контрольное чтение CSV не совпало с первичным")
                data_columns=list(test.columns)
                del test
                gc.collect()

                csv_path=str(parsed_path)
                used_name=filename
                ok=True

                # structure.csv — полезен для осмысленного отбора переменных,
                # но его отказ не аннулирует успешно полученные микроданные.
                try:
                    sb,su,st=_rosstat_download(
                        info["page"],info["structure"],timeout=25
                    )
                    sfile=struct_dir/f"structure_{year}.csv"
                    sfile.write_bytes(sb)
                    sdf=_read_structure(sb)
                    smap=_structure_mapping(sdf,data_columns)
                    if len(smap):
                        smap.insert(0,"year",year)
                        schema_parts.append(smap)
                        smap.to_excel(
                            struct_dir/f"structure_map_{year}.xlsx",index=False
                        )
                    structure_ok=True
                except Exception:
                    structure_ok=False

                break
            except Exception as e:
                err=repr(e)[:800]
                gc.collect()

        year_rows.append({
            "year":year,"ok":ok,"rows":nr,"cols":nc,
            "filename":used_name,"transport":transport,"url":used_url,
            "csv_path":csv_path,"structure_ok":structure_ok,
            "error":"" if ok else err
        })

        if ok:
            total_rows+=nr
            max_cols=max(max_cols,nc)

    year_audit=pd.DataFrame(year_rows)
    year_audit.to_excel(LOADER/"ROSSTAT_YEAR_AUDIT.xlsx",index=False)

    if schema_parts:
        schema=pd.concat(schema_parts,ignore_index=True)
        schema.to_excel(LOADER/"ROSSTAT_VARIABLE_DICTIONARY.xlsx",index=False)
    else:
        schema=pd.DataFrame(columns=["year","variable","description"])

    good=year_audit[year_audit["ok"].eq(True)].copy()
    if len(good):
        row["microdata_ok"]=True
        row["data_files"]=int(len(good))
        row["rows"]=int(total_rows)
        row["cols_max"]=int(max_cols)
        row["tables"]=int(len(schema))
        row["note"]=(
            "Успешно: "+", ".join(map(str,sorted(good.year.astype(int).tolist())))
            + ". Официальные CSV используются напрямую; общий raw parquet не создаётся."
        )
    else:
        row["note"]="Ни один год Росстата не прочитан; см. ROSSTAT_YEAR_AUDIT.xlsx"

    pbar.update(1)
    return row

def build_microdata_index():
    rows=[]
    root=DATA/"WCIOM"/"extracted"
    for source in WCIOM_PAGES:
        folder=root/source
        if not folder.exists():
            continue
        for p in candidate_data_files(folder):
            nr,nc,fmt=inspect_microdata(p)
            if nr>0 and nc>0:
                rows.append({
                    "source":source,"file":str(p),"format":fmt,
                    "rows":nr,"cols":nc,"size_mb":p.stat().st_size/1024**2
                })
    return pd.DataFrame(rows)


In [ ]:
# A.3 ЕДИНЫЙ ЗАПУСК ЗАГРУЗЧИКА — затем автоматический переход к эксперименту
LOAD_STEPS=8
load_rows=[]
t0=time.perf_counter()

with tqdm(total=LOAD_STEPS,desc="ЗАГРУЗЧИК ГЛАВЫ IV",bar_format=BAR_FMT,dynamic_ncols=True) as pbar:
    for source,url in WCIOM_PAGES.items():
        before=pbar.n
        try:
            load_rows.append(load_one_wciom(source,url,pbar))
        except Exception as e:
            load_rows.append({
                "source":source,"provider":"ВЦИОМ","page":url,
                "page_ok":False,"microdata_ok":False,"tables":0,
                "data_files":0,"rows":0,"cols_max":0,"note":repr(e)[:300]
            })
            if pbar.n-before<2:
                pbar.update(2-(pbar.n-before))

    before=pbar.n
    try:
        load_rows.append(load_rosstat_ict(pbar))
    except Exception as e:
        load_rows.append({
            "source":"rosstat_ict_2022_2025","provider":"Росстат",
            "page":ROSSTAT_ICT_SOURCES[2025]["page"],
            "page_ok":False,"microdata_ok":False,"tables":0,
            "data_files":0,"rows":0,"cols_max":0,"note":repr(e)[:500]
        })
        if pbar.n-before<2:
            pbar.update(2-(pbar.n-before))

    if pbar.n<pbar.total:
        pbar.update(pbar.total-pbar.n)

LOAD_AUDIT=pd.DataFrame(load_rows)
MICRODATA_INDEX=build_microdata_index()
LOAD_AUDIT["sec_total"]=round(time.perf_counter()-t0,1)

LOAD_AUDIT.to_excel(LOADER/"LOAD_AUDIT.xlsx",index=False)
MICRODATA_INDEX.to_excel(LOADER/"MICRODATA_INDEX.xlsx",index=False)

print(f"Загрузка завершена за {time.perf_counter()-t0:.1f} с.")
display(LOAD_AUDIT[[
    "source","provider","page_ok","microdata_ok","tables",
    "data_files","rows","cols_max","note"
]])

rap=LOADER/"ROSSTAT_YEAR_AUDIT.xlsx"
if rap.exists():
    print("Росстат — диагностика по годам:")
    display(pd.read_excel(rap))

print("Микроданные ВЦИОМ:")
display(MICRODATA_INDEX)

w_ok=int(LOAD_AUDIT.query("provider == 'ВЦИОМ'")["microdata_ok"].sum())
if w_ok < 3:
    raise RuntimeError(
        f"Получено только {w_ok}/3 массивов ВЦИОМ. "
        "Эксперимент автоматически остановлен: сначала нужен полный основной корпус."
    )

print("Основные массивы загружены. Эксперимент запускается автоматически.")


ЗАГРУЗЧИК ГЛАВЫ IV:   0%|          | 0/8 [00:00<?] 

Загрузка завершена за 72.0 с.


,source,provider,page_ok,microdata_ok,tables,data_files,rows,cols_max,note
0,traditional_values,ВЦИОМ,True,True,13,1,1605,37,
1,homeland,ВЦИОМ,True,True,4,1,1603,25,
2,ai_ethics,ВЦИОМ,True,True,10,1,1601,18,
3,rosstat_ict_2022_2025,Росстат,True,True,865,4,617401,242,"Успешно: 2022, 2023, 2024, 2025. Официальные C..."


Росстат — диагностика по годам:


,year,ok,rows,cols,filename,transport,url,csv_path,structure_ok,error
0,2025,True,154338,206,data-20260227T1302-structure-20260227T1302.csv,requests_noverify,https://rosstat.gov.ru/opendata/7708234640-IKT...,/content/MONOGRAPH_CHAPTER4_FINAL/DATA/ROSSTAT...,True,NaN
1,2024,True,154356,212,data-20250228T2002-structure-20250228T2002.zip,requests_noverify,https://rosstat.gov.ru/opendata/7708234640-IKT...,/content/MONOGRAPH_CHAPTER4_FINAL/DATA/ROSSTAT...,True,NaN
2,2023,True,154326,205,data-20240410T1504-structure-20240410T1504.zip,requests_noverify,https://rosstat.gov.ru/opendata/7708234640-IKT...,/content/MONOGRAPH_CHAPTER4_FINAL/DATA/ROSSTAT...,True,NaN
3,2022,True,154381,242,data-20230525T1005-structure-20230525T1005.zip,requests_noverify,https://rosstat.gov.ru/opendata/7708234640-IKT...,/content/MONOGRAPH_CHAPTER4_FINAL/DATA/ROSSTAT...,True,NaN


Микроданные ВЦИОМ:


,source,file,format,rows,cols,size_mb
0,traditional_values,/content/MONOGRAPH_CHAPTER4_FINAL/DATA/WCIOM/e...,sav,1605,37,0.138681
1,homeland,/content/MONOGRAPH_CHAPTER4_FINAL/DATA/WCIOM/e...,sav,1603,25,0.104468
2,ai_ethics,/content/MONOGRAPH_CHAPTER4_FINAL/DATA/WCIOM/e...,sav,1601,18,0.084067


Основные массивы загружены. Эксперимент запускается автоматически.


# Фаза B. Финальный анализ

In [ ]:

# B.1 NLP: строгая подготовка, устойчивые кластеры, макротипы и публикационные графики

from collections import defaultdict
from itertools import combinations
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score
import textwrap as _tw

NLP_DOMAINS = {
    "traditional_values":{"survey":"traditional_values","cols":["sem1_1d","sem1_2d","sem1_3d"],"label":"Традиционные ценности"},
    "ideal_family":{"survey":"traditional_values","cols":["sem3_1d","sem3_2d","sem3_3d"],"label":"Идеальная семья"},
    "homeland_meaning":{"survey":"homeland","cols":["rdn1_1d","rdn1_2d","rdn1_3d"],"label":"Смысл Родины"},
    "russia_associations":{"survey":"homeland","cols":["rdn2_1d","rdn2_2d","rdn2_3d"],"label":"Ассоциации с Россией"},
}
GENERIC_RE = re.compile(r"^(другое.*|затрудняюсь.*|нет таких.*|нет ассоциац.*|не знаю.*|нет ответа.*|отказ.*)$", re.I)
HEX_ID_RE = re.compile(r"^[0-9a-f]{12,}$", re.I)

def read_survey(path):
    if path.suffix.lower()==".sav":
        return pyreadstat.read_sav(path,apply_value_formats=True,formats_as_category=False)
    if path.suffix.lower()==".csv": return read_csv_auto(path),None
    if path.suffix.lower() in {".xlsx",".xls"}: return pd.read_excel(path),None
    raise ValueError(path)

def select_primary_files(index):
    out={}
    for source,g in index.groupby("source"):
        r=g.sort_values(["rows","cols","size_mb"],ascending=False).iloc[0]
        out[source]=Path(r["file"])
    return out

def label_map(meta):
    return (getattr(meta,"column_names_to_labels",{}) or {}) if meta is not None else {}

def detect_weight(df):
    if "weight1" in df.columns:
        w=pd.to_numeric(df["weight1"],errors="coerce")
        if w.notna().mean()>.9 and (w.fillna(0)>0).mean()>.9: return "weight1"
    return None

def age_group(s):
    x=pd.to_numeric(s,errors="coerce")
    return pd.cut(x,[17,24,34,44,54,64,200],labels=["18–24","25–34","35–44","45–54","55–64","65+"]).astype("string")

def internet_group(s):
    x=s.astype("string")
    out=pd.Series(pd.NA,index=x.index,dtype="string")
    out[x.str.contains("Не пользуюсь",case=False,na=False)]="Не пользуются"
    out[x.str.contains("более четырех",case=False,na=False)]="Ежедневно >4 ч"
    out[x.str.contains("менее четырех",case=False,na=False)]="Ежедневно <4 ч"
    out[x.str.contains("несколько раз",case=False,na=False)]="Реже ежедневно"
    return out

def weighted_dist(df,var,weight="weight1"):
    cols=[var]+([weight] if weight in df.columns else [])
    z=df[cols].dropna(subset=[var]).copy()
    if z.empty:return pd.DataFrame()
    w=pd.to_numeric(z[weight],errors="coerce").fillna(0) if weight in z.columns else pd.Series(1.,index=z.index)
    g=(pd.DataFrame({"category":z[var].astype(str),"w":w})
       .groupby("category",as_index=False).agg(weighted_n=("w","sum"),n=("category","size")))
    g["pct"]=100*g.weighted_n/g.weighted_n.sum()
    return g.sort_values("pct",ascending=False)

def clean_category(v):
    if pd.isna(v):return None
    s=re.sub(r"\s+"," ",str(v)).strip()
    if not s or GENERIC_RE.match(s) or HEX_ID_RE.match(s):return None
    return s

def build_semantic_documents(df,domain):
    spec=NLP_DOMAINS[domain]; cols=[c for c in spec["cols"] if c in df.columns]
    if not cols:return pd.DataFrame()
    wcol=detect_weight(df); rec=[]
    for pos,(_,r) in enumerate(df.iterrows()):
        cats=[]
        for c in cols:
            v=clean_category(r.get(c))
            if v and v not in cats:cats.append(v)
        if not cats:continue
        rid=str(r.get("ID",pos))
        rec.append({
            "respondent_id":rid,"domain":domain,"document":" ; ".join(cats),"categories":cats,
            "weight":float(pd.to_numeric(pd.Series([r.get(wcol,1.0)]),errors="coerce").fillna(1).iloc[0]) if wcol else 1.0,
            "AGE":r.get("AGE"),"age_group":age_group(pd.Series([r.get("AGE")])).iloc[0],
            "SEX":r.get("SEX"),"EDU":r.get("EDU"),"TIP":r.get("TIP")
        })
    out=pd.DataFrame(rec)
    if len(out):
        bad=out.document.str.fullmatch(r"[0-9a-f]{12,}",case=False,na=False)
        out=out.loc[~bad].reset_index(drop=True)
    return out

def load_encoder():
    from sentence_transformers import SentenceTransformer
    import torch
    device="cuda" if torch.cuda.is_available() else "cpu"
    models=["intfloat/multilingual-e5-large","sentence-transformers/paraphrase-multilingual-mpnet-base-v2"]
    last=None
    for name in models:
        try:
            enc=SentenceTransformer(name,device=device)
            return enc,name,device
        except Exception as e:
            last=e
    raise RuntimeError(f"Не удалось загрузить multilingual encoder: {last}")

def doc_hash(texts):
    h=hashlib.sha256()
    for x in texts:h.update((x+"\n").encode("utf-8"))
    return h.hexdigest()

def embed_domain(encoder,model_name,docs,domain,device):
    texts=["passage: "+x for x in docs.document.astype(str)]
    sig=doc_hash(texts)
    npy=CACHE/f"{domain}_embeddings.npy"; js=CACHE/f"{domain}_embeddings.json"
    if npy.exists() and js.exists():
        meta=json.loads(js.read_text(encoding="utf-8"))
        if meta.get("hash")==sig and meta.get("model")==model_name:
            E=np.load(npy)
            if len(E)==len(docs): return E
    E=encoder.encode(texts,batch_size=48 if device=="cuda" else 16,show_progress_bar=False,
                     normalize_embeddings=True,convert_to_numpy=True).astype("float32")
    np.save(npy,E)
    js.write_text(json.dumps({"hash":sig,"model":model_name,"n":len(docs)},ensure_ascii=False),encoding="utf-8")
    return E

def _stability(E,k,runs=8):
    labs=[KMeans(k,n_init=15,random_state=SEED+i).fit_predict(E) for i in range(runs)]
    vals=[adjusted_rand_score(labs[a],labs[b]) for a,b in combinations(range(runs),2)]
    return float(np.mean(vals)),float(np.min(vals)),float(np.max(vals))

def choose_k(E,k_values,min_share):
    n=len(E); ids=np.random.RandomState(SEED).choice(n,min(1200,n),replace=False)
    rows=[]; candidates=[]
    for k in k_values:
        if k>=n:continue
        lab=KMeans(k,n_init=25,random_state=SEED).fit_predict(E)
        cnt=np.bincount(lab,minlength=k); share=cnt.min()/n
        sil=silhouette_score(E[ids],lab[ids],metric="cosine")
        ari_mean,ari_min,ari_max=_stability(E,k,runs=6)
        valid=share>=min_share
        rows.append({"k":k,"silhouette":sil,"min_cluster_share":share,"ari_mean":ari_mean,"ari_min":ari_min,"valid":valid})
        if valid:candidates.append((sil,k,lab))
    if not candidates:
        candidates=[(r["silhouette"],r["k"],KMeans(r["k"],n_init=25,random_state=SEED).fit_predict(E)) for r in rows]
    best=max(candidates,key=lambda x:x[0])
    return best[1],best[2],pd.DataFrame(rows)

def weighted_enrichment(docs,labels):
    z=docs.copy();z["cluster"]=labels
    cats=sorted({c for xs in z.categories for c in xs});tw=z.weight.sum()
    p_all={c:z.loc[z.categories.map(lambda a:c in a),"weight"].sum()/tw for c in cats}
    rows=[]
    for k,g in z.groupby("cluster"):
        den=g.weight.sum()
        for c in cats:
            wc=g.loc[g.categories.map(lambda a:c in a),"weight"].sum()
            pc=wc/den if den else 0; pa=p_all[c]
            if pc<.025:continue
            lift=(pc+1e-4)/(pa+1e-4);score=pc*np.log2(lift)
            rows.append({"cluster":int(k),"category":c,"prevalence_cluster":pc,"prevalence_overall":pa,"lift":lift,"enrichment_score":score})
    return pd.DataFrame(rows).sort_values(["cluster","enrichment_score"],ascending=[True,False])

def short_cat(s,n=42):
    s=str(s).split("/")[0].strip()
    return s if len(s)<=n else s[:n-1]+"…"

def cluster_labels(enrichment,clusters,top_n=2):
    out={}
    for k in sorted(clusters):
        q=enrichment[(enrichment.cluster==k)&(enrichment.enrichment_score>0)].head(top_n)
        vals=[short_cat(x) for x in q.category]
        out[k]=" · ".join(vals) if vals else f"Тип {k+1}"
    return out

def prototypes(docs,E,labels,centers,n=5):
    rows=[]
    for k in sorted(set(labels)):
        ids=np.where(labels==k)[0];sims=E[ids]@centers[k]
        for rank,j in enumerate(ids[np.argsort(sims)[-n:][::-1]],1):
            rows.append({"cluster":int(k),"rank":rank,"respondent_id":docs.iloc[j].respondent_id,
                         "document":docs.iloc[j].document,"similarity":float(E[j]@centers[k])})
    return pd.DataFrame(rows)

def cluster_summary(docs,labels,label_map_):
    z=docs.copy();z["cluster"]=labels
    rows=[]
    for k,g in z.groupby("cluster"):
        rows.append({"cluster":int(k),"cluster_label":label_map_[k],"n":len(g),
                     "weighted_share_pct":100*g.weight.sum()/z.weight.sum()})
    return pd.DataFrame(rows).sort_values("weighted_share_pct",ascending=False)

def association_matrix(docs,top_n=15):
    node=defaultdict(float);edge=defaultdict(float)
    for _,r in docs.iterrows():
        cats=sorted(set(r.categories));w=float(r.weight)
        for c in cats:node[c]+=w
        for a,b in combinations(cats,2):edge[(a,b)]+=w
    cats=[x for x,_ in sorted(node.items(),key=lambda t:t[1],reverse=True)[:top_n]]
    M=np.eye(len(cats))
    for i,a in enumerate(cats):
        for j,b in enumerate(cats):
            if i==j:continue
            key=(a,b) if a<b else (b,a)
            co=edge.get(key,0.0);den=node[a]+node[b]-co
            M[i,j]=co/den if den else 0
    return cats,M

def plot_macro(summary,domain,path):
    s=summary.sort_values("weighted_share_pct")
    labels=[_tw.fill(x,36) for x in s.cluster_label]
    plt.figure(figsize=(10,5.5))
    plt.barh(labels,s.weighted_share_pct)
    plt.xlabel("Взвешенная доля, %");plt.ylabel("")
    plt.title(NLP_DOMAINS[domain]["label"]+": публикационные макротипы")
    for i,v in enumerate(s.weighted_share_pct):plt.text(v+.4,i,f"{v:.1f}%",va="center",fontsize=9)
    plt.tight_layout();plt.savefig(path,dpi=220,bbox_inches="tight");plt.close()

def plot_assoc_heatmap(cats,M,domain,path):
    labs=[_tw.fill(short_cat(x,28),20) for x in cats]
    plt.figure(figsize=(10,9))
    plt.imshow(M,aspect="auto",vmin=0,vmax=max(.05,float(np.nanmax(M))))
    plt.colorbar(label="Взвешенный Jaccard")
    plt.xticks(range(len(cats)),labs,rotation=60,ha="right",fontsize=7)
    plt.yticks(range(len(cats)),labs,fontsize=7)
    plt.title("Совместная встречаемость: "+NLP_DOMAINS[domain]["label"])
    plt.tight_layout();plt.savefig(path,dpi=200,bbox_inches="tight");plt.close()

def semantic_alignment(dfA,EA,dfB,EB,reps=400,boots=400):
    ia={x:i for i,x in enumerate(dfA.respondent_id)};ib={x:i for i,x in enumerate(dfB.respondent_id)}
    ids=sorted(set(ia)&set(ib))
    if len(ids)<100:return None,None,None
    a=np.array([ia[x] for x in ids]);b=np.array([ib[x] for x in ids])
    sim=np.sum(EA[a]*EB[b],axis=1);w=np.sqrt(dfA.iloc[a].weight.to_numpy()*dfB.iloc[b].weight.to_numpy())
    def wm(x,ww):return float(np.sum(x*ww)/np.sum(ww))
    obs=wm(sim,w);rng=np.random.RandomState(SEED)
    null=np.array([wm(np.sum(EA[a]*EB[rng.permutation(b)],axis=1),w) for _ in range(reps)])
    bs=np.array([wm(sim[ix:=rng.randint(0,len(sim),len(sim))],w[ix]) for _ in range(boots)])
    summ={"n":len(ids),"weighted_mean_similarity":obs,"bootstrap_ci_low":np.quantile(bs,.025),
          "bootstrap_ci_high":np.quantile(bs,.975),"random_mean":null.mean(),"random_sd":null.std(ddof=1),
          "delta_vs_random":obs-null.mean(),"standardized_delta":(obs-null.mean())/null.std(ddof=1) if null.std(ddof=1)>0 else np.nan,
          "permutation_p":(np.sum(null>=obs)+1)/(len(null)+1)}
    detail=pd.DataFrame({"respondent_id":ids,"cosine_similarity":sim,"weight":w})
    return summ,detail,null

def plot_alignment_null(null,obs,label,path):
    plt.figure(figsize=(8,5))
    plt.hist(null,bins=30,alpha=.75)
    plt.axvline(obs,linewidth=2,label=f"Наблюдаемое: {obs:.4f}")
    plt.axvline(np.mean(null),linestyle="--",linewidth=1.5,label=f"Permutation mean: {np.mean(null):.4f}")
    plt.xlabel("Средняя cosine similarity");plt.ylabel("Частота")
    plt.title(label);plt.legend();plt.tight_layout();plt.savefig(path,dpi=220,bbox_inches="tight");plt.close()


In [ ]:

# B.2 ML: разделение discrimination и probability/calibration

from scipy.special import softmax
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.preprocessing import OneHotEncoder, StandardScaler, SplineTransformer
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, balanced_accuracy_score, log_loss, confusion_matrix
from catboost import CatBoostClassifier, Pool

FEATURE_LABELS={
    "SEX":"Пол","AGE":"Возраст","FO":"Федеральный округ","TIP":"Тип поселения","EDU":"Образование",
    "DOHOD_0":"Материальное положение","PROF1":"Занятость / профессия","PROF2":"Профессиональный статус 2",
    "PROF3":"Профессиональный статус 3","d1":"Интенсивность использования Интернета","TV":"Телевизионное потребление"
}
CORE_FEATURES=["SEX","AGE","FO","TIP","EDU","DOHOD_0","PROF1"]
EXT_FEATURES=CORE_FEATURES+["PROF2","PROF3","d1","TV"]

def map_target(series,task):
    s=series.astype("string");out=pd.Series(pd.NA,index=s.index,dtype="string")
    if task=="ii1":
        out[s.str.contains("во всех возможных",case=False,na=False)]="Широкое применение"
        out[s.str.contains("только в некоторых",case=False,na=False)]="Селективное применение"
        out[s.str.contains("не должен использоваться вообще",case=False,na=False)]="Отказ от применения"
    elif task=="ii4":
        out[s.str.fullmatch(r"(?i)да|скорее да",na=False)]="Поддержка маркировки"
        out[s.str.fullmatch(r"(?i)нет|скорее нет",na=False)]="Неподдержка маркировки"
    elif task=="ii5":
        out[s.str.contains("только как помощника",case=False,na=False)]="Human-in-the-loop"
        out[s.str.contains("не следует привлекать",case=False,na=False)]="Без участия ИИ"
        out[s.str.contains("самостоятельно принимать решения",case=False,na=False)]="Автономное решение ИИ"
    return out

def prepare_ml_frame(df,target_var,task,features):
    cols=[c for c in features if c in df.columns]
    z=df[cols+[target_var]+(["weight1"] if "weight1" in df.columns else [])].copy()
    z["target"]=map_target(z[target_var],task);z=z.dropna(subset=["target"]).reset_index(drop=True)
    z["AGE"]=pd.to_numeric(z["AGE"],errors="coerce")
    z["weight"]=pd.to_numeric(z.get("weight1",1.0),errors="coerce").fillna(1.0).clip(lower=.001)
    z["age_group"]=age_group(z["AGE"])
    z["internet_group"]=internet_group(z["d1"]) if "d1" in z.columns else pd.NA
    return z,cols

def make_logit(features,balanced=False):
    age=["AGE"] if "AGE" in features else [];cats=[c for c in features if c!="AGE"]
    tr=[]
    if age:
        tr.append(("age",Pipeline([("imp",SimpleImputer(strategy="median")),
                                  ("spl",SplineTransformer(n_knots=5,degree=3,include_bias=False)),
                                  ("sc",StandardScaler())]),age))
    if cats:
        tr.append(("cat",Pipeline([("imp",SimpleImputer(strategy="most_frequent")),
                                  ("oh",OneHotEncoder(handle_unknown="infrequent_if_exist",min_frequency=10))]),cats))
    return Pipeline([("pre",ColumnTransformer(tr)),
                     ("clf",LogisticRegression(max_iter=4000,class_weight="balanced" if balanced else None,solver="lbfgs"))])

def prepare_catboost_X(z,features):
    X=z[features].copy();cat=[]
    for c in features:
        if c=="AGE":
            X[c]=pd.to_numeric(X[c],errors="coerce");X[c]=X[c].fillna(X[c].median())
        else:
            X[c]=X[c].fillna("NA").astype(str);cat.append(c)
    return X.reset_index(drop=True),cat,[X.columns.get_loc(c) for c in cat]

def align_proba(p,model_classes,classes):
    out=np.zeros((len(p),len(classes)),float);pos={str(c):i for i,c in enumerate(classes)}
    for j,c in enumerate(model_classes):
        if str(c) in pos:out[:,pos[str(c)]]=p[:,j]
    return out/np.clip(out.sum(axis=1,keepdims=True),1e-12,None)

def wmean(x,w):
    x=np.asarray(x,float);w=np.asarray(w,float);m=np.isfinite(x)&np.isfinite(w)&(w>=0)
    return float(np.sum(x[m]*w[m])/np.sum(w[m])) if m.any() and np.sum(w[m])>0 else np.nan

def multiclass_brier(y,p,classes,w):
    Y=np.column_stack([(np.asarray(y)==c).astype(float) for c in classes])
    return wmean(np.sum((p-Y)**2,axis=1),w)

def metrics(y,p,classes,w):
    pred=np.asarray(classes)[np.argmax(p,axis=1)]
    return {"macro_f1":f1_score(y,pred,average="macro",sample_weight=w,zero_division=0),
            "balanced_accuracy":balanced_accuracy_score(y,pred,sample_weight=w),
            "log_loss":log_loss(y,p,labels=list(classes),sample_weight=w),
            "brier":multiclass_brier(y,p,classes,w)}

def run_cv_final(z,features,task_name,repeats=2):
    Xraw=z[features].copy();y=z.target.astype(str).to_numpy();w=z.weight.to_numpy(float)
    classes=np.array(sorted(pd.unique(y)));K=len(classes);n=len(y)
    names=["WeightedPrior","SplineLogitBalanced","CatBoostBalanced","SplineLogitSurvey","CatBoostSurvey"]
    sums={m:np.zeros((n,K),float) for m in names};counts=np.zeros(n,int);fold_rows=[];fi=[]
    Xcb,cat,cat_idx=prepare_catboost_X(z,features)
    cv=RepeatedStratifiedKFold(n_splits=5,n_repeats=repeats,random_state=SEED)
    for fold,(tr,te) in enumerate(cv.split(Xraw,y),1):
        ytr,yte=y[tr],y[te];wtr,wte=w[tr],w[te]
        pri=np.array([wtr[ytr==c].sum() for c in classes]);pri=pri/pri.sum();p0=np.tile(pri,(len(te),1))
        lrb=make_logit(features,True);lrb.fit(Xraw.iloc[tr],ytr,clf__sample_weight=wtr)
        plrb=align_proba(lrb.predict_proba(Xraw.iloc[te]),lrb.named_steps["clf"].classes_,classes)
        lrs=make_logit(features,False);lrs.fit(Xraw.iloc[tr],ytr,clf__sample_weight=wtr)
        plrs=align_proba(lrs.predict_proba(Xraw.iloc[te]),lrs.named_steps["clf"].classes_,classes)
        loss="MultiClass" if K>2 else "Logloss"
        cbb=CatBoostClassifier(iterations=450,depth=5,learning_rate=.04,l2_leaf_reg=6,loss_function=loss,
                               auto_class_weights="Balanced",random_seed=SEED+fold,verbose=False,thread_count=-1)
        cbb.fit(Xcb.iloc[tr],ytr,cat_features=cat_idx,sample_weight=wtr)
        pcbb=align_proba(cbb.predict_proba(Xcb.iloc[te]),cbb.classes_,classes)
        cbs=CatBoostClassifier(iterations=450,depth=5,learning_rate=.04,l2_leaf_reg=6,loss_function=loss,
                               random_seed=SEED+fold,verbose=False,thread_count=-1)
        cbs.fit(Xcb.iloc[tr],ytr,cat_features=cat_idx,sample_weight=wtr)
        pcbs=align_proba(cbs.predict_proba(Xcb.iloc[te]),cbs.classes_,classes)
        probs={"WeightedPrior":p0,"SplineLogitBalanced":plrb,"CatBoostBalanced":pcbb,"SplineLogitSurvey":plrs,"CatBoostSurvey":pcbs}
        for name,p in probs.items():
            sums[name][te]+=p;fold_rows.append({"task":task_name,"fold":fold,"model":name,**metrics(yte,p,classes,wte)})
        for feat,val in zip(features,cbb.get_feature_importance()):
            fi.append({"task":task_name,"fold":fold,"feature":feat,"importance":float(val)})
        counts[te]+=1
    prob={m:sums[m]/counts[:,None] for m in names}
    overall=pd.DataFrame([{"task":task_name,"model":m,**metrics(y,p,classes,w)} for m,p in prob.items()])
    return {"classes":classes,"y":y,"w":w,"prob":prob,"overall":overall,
            "fold_metrics":pd.DataFrame(fold_rows),"feature_importance_folds":pd.DataFrame(fi),"Xraw":Xraw}

def bootstrap_metrics(cvres,B=250):
    y,w,classes=cvres["y"],cvres["w"],cvres["classes"];rng=np.random.RandomState(SEED);rows=[]
    for model,p in cvres["prob"].items():
        base=metrics(y,p,classes,w);store=defaultdict(list)
        for _ in range(B):
            ix=rng.randint(0,len(y),len(y))
            if len(np.unique(y[ix]))<len(classes):continue
            mm=metrics(y[ix],p[ix],classes,w[ix])
            for k,v in mm.items():store[k].append(v)
        for k,a in store.items():
            lo,hi=np.quantile(a,[.025,.975]);rows.append({"model":model,"metric":k,"estimate":base[k],"ci_low":lo,"ci_high":hi})
    return pd.DataFrame(rows)

def choose_probability_model(cvres):
    t=cvres["overall"].set_index("model")
    candidates=["SplineLogitSurvey","CatBoostSurvey"]
    return min(candidates,key=lambda m:(t.loc[m,"log_loss"],t.loc[m,"brier"]))

def calibration_table(cvres,model,bins=8):
    y,p,classes,w=cvres["y"],cvres["prob"][model],cvres["classes"],cvres["w"];rows=[]
    for j,c in enumerate(classes):
        q=pd.Series(p[:,j]).rank(method="first")
        b=pd.qcut(q,q=min(bins,len(q)),labels=False,duplicates="drop")
        tmp=pd.DataFrame({"bin":b,"pred":p[:,j],"obs":(y==c).astype(float),"w":w})
        for bi,g in tmp.groupby("bin"):
            rows.append({"class":c,"bin":int(bi),"predicted":wmean(g.pred,g.w),"observed":wmean(g.obs,g.w),"n":len(g)})
    return pd.DataFrame(rows)

def observed_profiles(z,group_vars):
    rows=[]
    for gv in group_vars:
        if gv not in z.columns:continue
        for val,g in z.groupby(gv,dropna=False):
            ww=g.weight.to_numpy(float);neff=(ww.sum()**2/np.sum(ww**2)) if np.sum(ww**2)>0 else np.nan
            for c in sorted(z.target.unique()):
                obs=wmean((g.target.to_numpy()==c).astype(float),ww)
                se=np.sqrt(max(obs*(1-obs),0)/neff) if np.isfinite(neff) and neff>0 else np.nan
                rows.append({"group_variable":gv,"group_value":str(val),"class":c,"n":len(g),"n_eff":neff,
                             "observed_pct":100*obs,"ci_low_pct":100*max(0,obs-1.96*se),"ci_high_pct":100*min(1,obs+1.96*se)})
    return pd.DataFrame(rows)

def safe_catboost_shap(z,features,model_path):
    z=z.reset_index(drop=True).copy();X,cat,cat_idx=prepare_catboost_X(z,features)
    y=z.target.astype(str).reset_index(drop=True);w=z.weight.to_numpy(float);K=y.nunique()
    cb=CatBoostClassifier(iterations=600,depth=5,learning_rate=.035,l2_leaf_reg=6,
                          loss_function="MultiClass" if K>2 else "Logloss",auto_class_weights="Balanced",
                          random_seed=SEED,verbose=False,thread_count=-1)
    cb.fit(X,y,cat_features=cat_idx,sample_weight=w);cb.save_model(str(model_path))
    rng=np.random.RandomState(SEED);pos=rng.choice(len(X),min(900,len(X)),replace=False)
    sample=X.iloc[pos].reset_index(drop=True);ys=y.iloc[pos].reset_index(drop=True);ws=w[pos]
    sv=np.asarray(cb.get_feature_importance(Pool(sample,ys,cat_features=cat_idx,weight=ws),type="ShapValues"))
    class_rows=[]
    if sv.ndim==3 and sv.shape[1]==len(cb.classes_):
        A=sv[:,:,:-1];glob=np.mean(np.abs(A),axis=(0,1))
        for ci,cl in enumerate(cb.classes_):
            for feat,val in zip(features,np.mean(np.abs(A[:,ci,:]),axis=0)):
                class_rows.append({"class":str(cl),"feature":feat,"mean_abs_shap":float(val)})
    elif sv.ndim==3 and sv.shape[2]==len(cb.classes_):
        A=np.moveaxis(sv[:,:-1,:],-1,1);glob=np.mean(np.abs(A),axis=(0,1))
        for ci,cl in enumerate(cb.classes_):
            for feat,val in zip(features,np.mean(np.abs(A[:,ci,:]),axis=0)):
                class_rows.append({"class":str(cl),"feature":feat,"mean_abs_shap":float(val)})
    elif sv.ndim==2:
        glob=np.mean(np.abs(sv[:,:-1]),axis=0)
    else:
        glob=np.asarray(cb.get_feature_importance())
    g=pd.DataFrame({"feature":features,"feature_label":[FEATURE_LABELS.get(x,x) for x in features],"mean_abs_shap":glob}).sort_values("mean_abs_shap",ascending=False)
    return cb,g,pd.DataFrame(class_rows)

def weighted_cramers_v(x,y,w):
    d=pd.DataFrame({"x":x.astype(str),"y":y.astype(str),"w":w}).dropna()
    tab=d.pivot_table(index="x",columns="y",values="w",aggfunc="sum",fill_value=0)
    if min(tab.shape)<2:return np.nan
    A=tab.to_numpy(float);n=A.sum();exp=A.sum(1,keepdims=True)@A.sum(0,keepdims=True)/n
    chi=np.nansum((A-exp)**2/np.where(exp==0,np.nan,exp))
    return float(np.sqrt((chi/n)/max(1,min(A.shape[0]-1,A.shape[1]-1))))

def plot_model_ci(ci,task,metric,path,title):
    d=ci[(ci.task==task)&(ci.metric==metric)&ci.model.isin(["WeightedPrior","SplineLogitBalanced","CatBoostBalanced"])].copy()
    order=["WeightedPrior","SplineLogitBalanced","CatBoostBalanced"];d["ord"]=d.model.map({x:i for i,x in enumerate(order)});d=d.sort_values("ord")
    labels={"WeightedPrior":"Базовая модель","SplineLogitBalanced":"Spline logit","CatBoostBalanced":"CatBoost"}
    x=np.arange(len(d));y=d.estimate.to_numpy();lo=y-d.ci_low.to_numpy();hi=d.ci_high.to_numpy()-y
    plt.figure(figsize=(8,5));plt.errorbar(x,y,yerr=[lo,hi],fmt="o",capsize=5)
    plt.xticks(x,[labels[x] for x in d.model]);plt.ylabel(metric);plt.title(title)
    plt.tight_layout();plt.savefig(path,dpi=220,bbox_inches="tight");plt.close()

def plot_calibration(tab,path,title):
    plt.figure(figsize=(7,6));plt.plot([0,1],[0,1],linestyle="--",linewidth=1,label="Идеальная калибровка")
    for c,g in tab.groupby("class"):
        g=g.sort_values("predicted");plt.plot(g.predicted,g.observed,marker="o",label=str(c))
    plt.xlabel("Средняя предсказанная вероятность");plt.ylabel("Наблюдаемая взвешенная доля")
    plt.title(title);plt.legend(fontsize=8);plt.tight_layout();plt.savefig(path,dpi=210,bbox_inches="tight");plt.close()

def plot_stacked_profiles(profiles,gv,order,path,title):
    d=profiles[profiles.group_variable.eq(gv)].copy();classes=["Отказ от применения","Селективное применение","Широкое применение"]
    piv=d.pivot(index="group_value",columns="class",values="observed_pct").reindex(order).fillna(0)
    plt.figure(figsize=(10,5.8));bottom=np.zeros(len(piv))
    for c in classes:
        vals=piv[c].to_numpy() if c in piv else np.zeros(len(piv))
        plt.bar(np.arange(len(piv)),vals,bottom=bottom,label=c);bottom+=vals
    plt.xticks(np.arange(len(piv)),[_tw.fill(str(x),18) for x in piv.index],rotation=25,ha="right")
    plt.ylabel("%");plt.ylim(0,100);plt.title(title);plt.legend(fontsize=8,ncol=3,loc="upper center",bbox_to_anchor=(.5,-.18))
    plt.tight_layout();plt.savefig(path,dpi=220,bbox_inches="tight");plt.close()

def plot_shap(shap_df,path):
    d=shap_df.head(10).sort_values("mean_abs_shap")
    plt.figure(figsize=(9,5.8));plt.barh(d.feature_label,d.mean_abs_shap)
    plt.xlabel("Среднее |SHAP|");plt.title("Факторы различимости отношения к применению ИИ")
    plt.tight_layout();plt.savefig(path,dpi=220,bbox_inches="tight");plt.close()


In [ ]:

# B.3 Росстат: ключевые цифровые временные ряды

def rosstat_context():
    audit_path=LOADER/"ROSSTAT_YEAR_AUDIT.xlsx"
    if not audit_path.exists():return pd.DataFrame(),pd.DataFrame()
    ya=pd.read_excel(audit_path);ya=ya[ya.ok.eq(True)].copy()
    dict_path=LOADER/"ROSSTAT_VARIABLE_DICTIONARY.xlsx"
    dictionary=pd.read_excel(dict_path) if dict_path.exists() else pd.DataFrame()
    schema=[];rows=[]
    for _,a in ya.iterrows():
        year=int(a.year);p=Path(str(a.csv_path))
        if not p.exists():continue
        df=read_csv_auto(p)
        dct=dictionary[dictionary.year.eq(year)] if len(dictionary) and "year" in dictionary.columns else pd.DataFrame()
        desc=dict(zip(dct.variable.astype(str),dct.description.astype(str))) if len(dct) else {}
        w=_coerce_numeric_series(df["VESA_DX"]).fillna(0) if "VESA_DX" in df.columns else pd.Series(1.,index=df.index)
        for c in df.columns:
            schema.append({"year":year,"variable":str(c),"description":desc.get(str(c),""),"nonmissing":int(df[c].notna().sum()),"n_unique":int(df[c].nunique(dropna=True))})
        for c in df.columns:
            lab=str(c)+" "+desc.get(str(c),"")
            if not re.search(r"интернет|смартф|мобиль|нейросет|электрон|онлайн|госуслуг|компьют",lab,re.I):continue
            s=df[c].map(_decode_scalar).astype("string")
            if not (2<=s.nunique(dropna=True)<=80):continue
            g=(pd.DataFrame({"value":s,"w":w}).dropna(subset=["value"]).groupby("value",as_index=False).agg(weighted_n=("w","sum"),n=("value","size")))
            if g.weighted_n.sum()<=0:continue
            g["pct"]=100*g.weighted_n/g.weighted_n.sum()
            for _,r in g.iterrows():
                rows.append({"year":year,"variable":str(c),"description":desc.get(str(c),""),"category":str(r.value),"pct":float(r.pct),"n":int(r.n)})
        del df
        gc.collect()
    return pd.DataFrame(schema),pd.DataFrame(rows)

def rosstat_key_trends(ctx):
    specs=[
        ("Доступ к Интернету дома","CInt1","1.0"),
        ("Широкополосный доступ","CInt3_E","1.0"),
        ("Использование смартфона","M1","1.0"),
    ]
    rows=[]
    for label,var,cat in specs:
        q=ctx[(ctx.variable==var)&(ctx.category.astype(str)==cat)]
        for _,r in q.iterrows():rows.append({"indicator":label,"year":int(r.year),"pct":float(r.pct)})
    q=ctx[(ctx.variable=="Int5_34")&(ctx.category.astype(str)!="0.0")]
    for _,r in q.iterrows():rows.append({"indicator":"Использование нейросетей","year":int(r.year),"pct":float(r.pct)})
    return pd.DataFrame(rows)

def plot_rosstat_connectivity(trends,path):
    d=trends[trends.indicator!="Использование нейросетей"]
    plt.figure(figsize=(8.5,5.5))
    for ind,g in d.groupby("indicator"):
        g=g.sort_values("year");plt.plot(g.year,g.pct,marker="o",label=ind)
    plt.xticks(sorted(d.year.unique()));plt.ylim(0,100);plt.ylabel("%");plt.xlabel("Год")
    plt.title("Распространение базовых цифровых практик, Росстат");plt.legend(fontsize=8)
    plt.tight_layout();plt.savefig(path,dpi=220,bbox_inches="tight");plt.close()

def plot_neural_use(trends,path):
    d=trends[trends.indicator=="Использование нейросетей"].sort_values("year")
    if d.empty:return
    plt.figure(figsize=(6.5,5));plt.bar(d.year.astype(str),d.pct)
    for i,v in enumerate(d.pct):plt.text(i,v+.2,f"{v:.1f}%",ha="center")
    plt.ylabel("%");plt.title("Использование нейросетей для создания контента")
    plt.tight_layout();plt.savefig(path,dpi=220,bbox_inches="tight");plt.close()


In [ ]:

# C.1 ФИНАЛЬНЫЙ ЭКСПЕРИМЕНТ — единый автоматический запуск

t0=time.perf_counter()
STEPS=12
surveys={};metas={}
weighted_tables=[]
fine_search=[];fine_summary=[];fine_stability=[];macro_search=[];macro_summary=[];macro_stability=[]
macro_enrichment=[];macro_prototypes=[];assoc_tables=[];alignment_rows=[];alignment_null_rows=[]
ml_audit=[];ml_overall=[];ml_fold=[];ml_ci=[];ml_fi=[];ml_shap_g=[];ml_shap_c=[];ml_profiles=[];ml_assoc=[];prob_select=[];calibration=[]
figure_index=[]

with tqdm(total=STEPS,desc="FINAL CHAPTER IV",bar_format=BAR_FMT,dynamic_ncols=True) as pb:
    # 1
    pb.set_postfix_str("1/12 микроданные")
    primary=select_primary_files(MICRODATA_INDEX)
    for source,path in primary.items():
        df,meta=read_survey(path);surveys[source]=df;metas[source]=meta
    pb.update(1)

    # 2
    pb.set_postfix_str("2/12 аудит / распределения")
    SURVEY_AUDIT=pd.DataFrame([{"source":s,"rows":len(d),"cols":len(d.columns),"weight_ok":"weight1" in d.columns} for s,d in surveys.items()])
    for source,vars_ in {"traditional_values":["sem2_1","sem9","gr1_1","gr1_2","gr1_3","gr1_4","gr1_5"],
                         "ai_ethics":["ii1","ii2","ii3","ii4","ii5","d1","TV"]}.items():
        for v in vars_:
            if source in surveys and v in surveys[source].columns:
                q=weighted_dist(surveys[source],v)
                if len(q):q.insert(0,"source",source);q.insert(1,"variable",v);weighted_tables.append(q)
    WEIGHTED_DISTRIBUTIONS=pd.concat(weighted_tables,ignore_index=True) if weighted_tables else pd.DataFrame()
    pb.update(1)

    # 3
    pb.set_postfix_str("3/12 NLP documents / embeddings")
    encoder,ENCODER_NAME,DEVICE=load_encoder()
    docs_by={};emb_by={}
    for domain,spec in tqdm(NLP_DOMAINS.items(),desc="NLP domains",leave=False):
        docs=build_semantic_documents(surveys[spec["survey"]],domain)
        if len(docs)<100:continue
        docs_by[domain]=docs;emb_by[domain]=embed_domain(encoder,ENCODER_NAME,docs,domain,DEVICE)
    pb.update(1)

    # 4 fine clustering
    pb.set_postfix_str("4/12 fine clustering")
    fine_labels={}
    for domain,docs in docs_by.items():
        E=emb_by[domain];k,lab,search=choose_k(E,range(3,11),.035);fine_labels[domain]=lab
        search.insert(0,"domain",domain);fine_search.append(search)
        enr=weighted_enrichment(docs,lab);labels_=cluster_labels(enr,sorted(set(lab)),2)
        sm=cluster_summary(docs,lab,labels_);sm.insert(0,"domain",domain);fine_summary.append(sm)
        st=search.loc[search.k.eq(k)].iloc[0]
        fine_stability.append({"domain":domain,"k":k,"silhouette":st.silhouette,"ari_mean":st.ari_mean,"ari_min":st.ari_min})
        # UMAP diagnostic only
        try:
            import umap
            xy=umap.UMAP(n_neighbors=25,min_dist=.08,n_components=2,metric="cosine",random_state=SEED).fit_transform(E)
            plt.figure(figsize=(8,6))
            for cl in sorted(set(lab)):
                m=lab==cl;plt.scatter(xy[m,0],xy[m,1],s=8,alpha=.45,label=str(cl))
            plt.title("Диагностика UMAP: "+NLP_DOMAINS[domain]["label"]);plt.legend(fontsize=7,ncol=2)
            plt.tight_layout();plt.savefig(DIAG/f"UMAP_{domain}.png",dpi=170);plt.close()
        except Exception:pass
    NLP_FINE_SEARCH=pd.concat(fine_search,ignore_index=True);NLP_FINE_SUMMARY=pd.concat(fine_summary,ignore_index=True);NLP_FINE_STABILITY=pd.DataFrame(fine_stability)
    pb.update(1)

    # 5 macro clustering + publication bars
    pb.set_postfix_str("5/12 публикационные макротипы")
    macro_labels_by={}
    for domain,docs in docs_by.items():
        E=emb_by[domain];k,lab,search=choose_k(E,range(3,6),.07)
        macro_labels_by[domain]=lab;search.insert(0,"domain",domain);macro_search.append(search)
        km=KMeans(k,n_init=40,random_state=SEED).fit(E);lab=km.labels_;macro_labels_by[domain]=lab
        enr=weighted_enrichment(docs,lab);labs=cluster_labels(enr,sorted(set(lab)),2)
        sm=cluster_summary(docs,lab,labs);sm.insert(0,"domain",domain);macro_summary.append(sm)
        enr.insert(0,"domain",domain);macro_enrichment.append(enr)
        pr=prototypes(docs,E,lab,km.cluster_centers_,5);pr.insert(0,"domain",domain);macro_prototypes.append(pr)
        st=search.loc[search.k.eq(k)].iloc[0]
        macro_stability.append({"domain":domain,"k":k,"silhouette":st.silhouette,"ari_mean":st.ari_mean,"ari_min":st.ari_min})
        f=PUB/f"PUB_NLP_{domain}_macrotypes.png";plot_macro(sm,domain,f)
        figure_index.append({"file":f.name,"status":"MAIN","purpose":"Публикационные макротипы "+NLP_DOMAINS[domain]["label"]})
    NLP_MACRO_SEARCH=pd.concat(macro_search,ignore_index=True);NLP_MACRO_SUMMARY=pd.concat(macro_summary,ignore_index=True)
    NLP_MACRO_STABILITY=pd.DataFrame(macro_stability);NLP_MACRO_ENRICHMENT=pd.concat(macro_enrichment,ignore_index=True)
    NLP_MACRO_PROTOTYPES=pd.concat(macro_prototypes,ignore_index=True)
    pb.update(1)

    # 6 association heatmaps
    pb.set_postfix_str("6/12 матрицы совместной встречаемости")
    for domain,docs in docs_by.items():
        cats,M=association_matrix(docs,15)
        for i,a in enumerate(cats):
            for j,b in enumerate(cats):
                if j>i:assoc_tables.append({"domain":domain,"category_a":a,"category_b":b,"weighted_jaccard":float(M[i,j])})
        f=DIAG/f"ASSOC_{domain}.png";plot_assoc_heatmap(cats,M,domain,f)
        figure_index.append({"file":f.name,"status":"APPENDIX","purpose":"Матрица смысловых ассоциаций"})
    NLP_ASSOCIATIONS=pd.DataFrame(assoc_tables)
    pb.update(1)

    # 7 semantic alignment
    pb.set_postfix_str("7/12 semantic alignment")
    for name,a,b,label in [("values_vs_family","traditional_values","ideal_family","Ценности ↔ идеальная семья"),
                           ("homeland_vs_russia","homeland_meaning","russia_associations","Родина ↔ Россия")]:
        if a not in docs_by or b not in docs_by:continue
        summ,detail,null=semantic_alignment(docs_by[a],emb_by[a],docs_by[b],emb_by[b])
        if summ:
            summ["pair"]=name
            summ["effect_interpretation"]="очень малый" if abs(summ["delta_vs_random"])<.01 else ("малый" if abs(summ["delta_vs_random"])<.03 else "заметный")
            alignment_rows.append(summ)
            for v in null:alignment_null_rows.append({"pair":name,"null_similarity":float(v)})
            f=PUB/f"PUB_SEMANTIC_{name}.png";plot_alignment_null(null,summ["weighted_mean_similarity"],label,f)
            figure_index.append({"file":f.name,"status":"MAIN","purpose":"Наблюдаемая семантическая связь против permutation baseline"})
    SEMANTIC_ALIGNMENT=pd.DataFrame(alignment_rows);SEMANTIC_NULL=pd.DataFrame(alignment_null_rows)
    pb.update(1)

    # 8 ML prepare
    pb.set_postfix_str("8/12 ML preparation")
    ai=surveys["ai_ethics"].copy()
    configs=[("ii1_core","ii1","ii1",CORE_FEATURES),("ii1_extended","ii1","ii1",EXT_FEATURES),
             ("ii4_extended","ii4","ii4",EXT_FEATURES),("ii5_extended","ii5","ii5",EXT_FEATURES)]
    ml_frames={}
    for name,target,task,features in configs:
        z,feats=prepare_ml_frame(ai,target,task,features);ml_frames[name]=(z,feats)
        cnt=z.target.value_counts()
        ml_audit.append({"task":name,"n":len(z),"classes":z.target.nunique(),"min_class_n":int(cnt.min()),
                         "rare_class_warning":bool(cnt.min()<80),"features":" | ".join(feats),
                         "class_distribution":" | ".join(f"{k}:{v}" for k,v in cnt.items())})
    ML_TASK_AUDIT=pd.DataFrame(ml_audit)
    pb.update(1)

    # 9 ML CV
    pb.set_postfix_str("9/12 repeated CV")
    mlres={}
    for name,_,_,_ in tqdm(configs,desc="ML tasks",leave=False):
        z,feats=ml_frames[name];res=run_cv_final(z,feats,name,2);mlres[name]=res
        ml_overall.append(res["overall"]);ml_fold.append(res["fold_metrics"])
        ci=bootstrap_metrics(res,250);ci.insert(0,"task",name);ml_ci.append(ci)
        fs=(res["feature_importance_folds"].groupby("feature").agg(importance_mean=("importance","mean"),importance_sd=("importance","std")).reset_index())
        fs.insert(0,"task",name);ml_fi.append(fs)
        pm=choose_probability_model(res);prob_select.append({"task":name,"probability_model":pm,
            "log_loss":float(res["overall"].set_index("model").loc[pm,"log_loss"]),
            "brier":float(res["overall"].set_index("model").loc[pm,"brier"])})
        cal=calibration_table(res,pm,8);cal.insert(0,"task",name);cal.insert(1,"model",pm);calibration.append(cal)
    ML_OVERALL=pd.concat(ml_overall,ignore_index=True);ML_FOLD=pd.concat(ml_fold,ignore_index=True)
    ML_CI=pd.concat(ml_ci,ignore_index=True);ML_FEATURE_STABILITY=pd.concat(ml_fi,ignore_index=True)
    ML_PROBABILITY_SELECTION=pd.DataFrame(prob_select);ML_CALIBRATION=pd.concat(calibration,ignore_index=True)
    pb.update(1)

    # 10 SHAP, observed profiles, associations, ML figures
    pb.set_postfix_str("10/12 SHAP / profiles / calibration")
    for name in ["ii1_extended","ii4_extended","ii5_extended"]:
        z,feats=ml_frames[name]
        _,sg,sc=safe_catboost_shap(z,feats,MODELS/f"{name}_catboost_balanced.cbm")
        sg.insert(0,"task",name);ml_shap_g.append(sg)
        if len(sc):sc.insert(0,"task",name);ml_shap_c.append(sc)
        gp=observed_profiles(z,["age_group","internet_group","EDU","TIP","SEX"]);gp.insert(0,"task",name);ml_profiles.append(gp)
        for gv in ["age_group","internet_group","EDU","TIP","SEX","DOHOD_0"]:
            if gv not in z.columns:continue
            ml_assoc.append({"task":name,"group_variable":gv,
                             "weighted_cramers_v":weighted_cramers_v(z[gv].astype("string"),z.target,z.weight.to_numpy(float))})
    ML_SHAP_GLOBAL=pd.concat(ml_shap_g,ignore_index=True);ML_SHAP_CLASS=pd.concat(ml_shap_c,ignore_index=True) if ml_shap_c else pd.DataFrame()
    ML_GROUP_PROFILES=pd.concat(ml_profiles,ignore_index=True);ML_ASSOCIATIONS=pd.DataFrame(ml_assoc)
    # publication ML figures
    plot_model_ci(ML_CI,"ii1_extended","macro_f1",PUB/"PUB_ML_macro_f1.png","Различимость установок к применению ИИ")
    figure_index.append({"file":"PUB_ML_macro_f1.png","status":"MAIN","purpose":"OOF macro-F1 с bootstrap 95% CI"})
    plot_model_ci(ML_CI,"ii1_extended","balanced_accuracy",PUB/"PUB_ML_balanced_accuracy.png","Balanced accuracy: отношение к применению ИИ")
    figure_index.append({"file":"PUB_ML_balanced_accuracy.png","status":"APPENDIX","purpose":"OOF balanced accuracy с bootstrap 95% CI"})
    plot_shap(ML_SHAP_GLOBAL[ML_SHAP_GLOBAL.task.eq("ii1_extended")],PUB/"PUB_ML_SHAP_ii1.png")
    figure_index.append({"file":"PUB_ML_SHAP_ii1.png","status":"MAIN","purpose":"Факторы различимости по balanced CatBoost"})
    gp=ML_GROUP_PROFILES[ML_GROUP_PROFILES.task.eq("ii1_extended")]
    plot_stacked_profiles(gp,"age_group",["18–24","25–34","35–44","45–54","55–64","65+"],PUB/"PUB_AI_BY_AGE.png","Отношение к применению ИИ по возрасту")
    figure_index.append({"file":"PUB_AI_BY_AGE.png","status":"MAIN","purpose":"Наблюдаемые survey-weighted различия по возрасту"})
    plot_stacked_profiles(gp,"internet_group",["Не пользуются","Реже ежедневно","Ежедневно <4 ч","Ежедневно >4 ч"],PUB/"PUB_AI_BY_INTERNET.png","Отношение к применению ИИ и интенсивность Интернета")
    figure_index.append({"file":"PUB_AI_BY_INTERNET.png","status":"MAIN","purpose":"Наблюдаемые survey-weighted различия по цифровой интенсивности"})
    pm=ML_PROBABILITY_SELECTION.loc[ML_PROBABILITY_SELECTION.task.eq("ii1_extended"),"probability_model"].iloc[0]
    plot_calibration(ML_CALIBRATION[ML_CALIBRATION.task.eq("ii1_extended")],DIAG/"CALIBRATION_ii1.png","Калибровка probability-модели: "+pm)
    figure_index.append({"file":"CALIBRATION_ii1.png","status":"APPENDIX","purpose":"Диагностика вероятностной калибровки"})
    pb.update(1)

    # 11 Rosstat
    pb.set_postfix_str("11/12 Росстат / временные ряды")
    ROSSTAT_SCHEMA,ROSSTAT_CONTEXT=rosstat_context();ROSSTAT_KEY_TRENDS=rosstat_key_trends(ROSSTAT_CONTEXT)
    plot_rosstat_connectivity(ROSSTAT_KEY_TRENDS,PUB/"PUB_ROSSTAT_CONNECTIVITY.png")
    figure_index.append({"file":"PUB_ROSSTAT_CONNECTIVITY.png","status":"MAIN","purpose":"Динамика базовых цифровых практик 2022–2025"})
    plot_neural_use(ROSSTAT_KEY_TRENDS,PUB/"PUB_ROSSTAT_NEURAL_USE.png")
    figure_index.append({"file":"PUB_ROSSTAT_NEURAL_USE.png","status":"MAIN","purpose":"Использование нейросетей 2024–2025"})
    pb.update(1)

    # 12 Interpretation + export
    pb.set_postfix_str("12/12 интерпретация / экспорт")
    FIGURE_INDEX=pd.DataFrame(figure_index)
    lines=["# Финальная интерпретация IV раздела","",
           "## Данные",
           f"ВЦИОМ: {len(surveys)} независимых массива, суммарно {sum(len(x) for x in surveys.values())} наблюдений.",
           f"Росстат: {ROSSTAT_SCHEMA.year.nunique() if len(ROSSTAT_SCHEMA) else 0} года микроданных ИКТ.","",
           "## NLP"]
    for _,r in NLP_MACRO_STABILITY.iterrows():
        lines.append(f"- {NLP_DOMAINS[r.domain]['label']}: {int(r.k)} макротипа/ов; silhouette={r.silhouette:.3f}; ARI={r.ari_mean:.3f}.")
    lines+=["","### Семантическое сопоставление"]
    for _,r in SEMANTIC_ALIGNMENT.iterrows():
        lines.append(f"- {r.pair}: observed={r.weighted_mean_similarity:.4f}; permutation mean={r.random_mean:.4f}; Δ={r.delta_vs_random:.4f}; p={r.permutation_p:.4f}; величина эффекта: {r.effect_interpretation}.")
    lines+=["","## ML: различимость и вероятность — разные задачи"]
    t=ML_OVERALL[ML_OVERALL.task.eq("ii1_extended")].set_index("model")
    lines.append(f"- Balanced CatBoost: macro-F1={t.loc['CatBoostBalanced','macro_f1']:.3f}, balanced accuracy={t.loc['CatBoostBalanced','balanced_accuracy']:.3f}.")
    lines.append(f"- Balanced spline-logit: macro-F1={t.loc['SplineLogitBalanced','macro_f1']:.3f}.")
    pm=ML_PROBABILITY_SELECTION.loc[ML_PROBABILITY_SELECTION.task.eq("ii1_extended"),"probability_model"].iloc[0]
    lines.append(f"- Для вероятностной интерпретации используется {pm}, выбранная по OOF log-loss/Brier среди survey-weighted небалансированных моделей.")
    lines.append("- Balanced CatBoost используется только для discrimination/SHAP; его class-balanced вероятности не используются как оценки долей.")
    rr=ML_TASK_AUDIT[ML_TASK_AUDIT.task.eq("ii5_extended")].iloc[0]
    if rr.rare_class_warning:lines.append(f"- ii5 трактуется как дополнительный анализ: минимальный класс n={int(rr.min_class_n)}, поэтому выводы об автономном решении ИИ ограничены.")
    lines+=["","## Содержательные ограничения",
            "1. Независимые опросы ВЦИОМ не объединяются на уровне респондентов.",
            "2. Статистическая значимость семантической связи не подменяет оценку величины эффекта.",
            "3. ML не используется для причинных выводов.",
            "4. SHAP отражает вклад признаков в различение классов, а не причинное влияние.",
            "5. Росстат служит внешней триангуляцией цифрового контекста.",
            "6. UMAP и association heatmaps имеют диагностический/иллюстративный статус; основные выводы опираются на таблицы, устойчивость и взвешенные распределения."]
    (RESULTS/"INTERPRETATION_FINAL.md").write_text("\n".join(lines),encoding="utf-8")

    EXPERIMENT_AUDIT=pd.DataFrame([{"status":"OK","encoder":ENCODER_NAME,"surveys_loaded":len(surveys),
        "respondents_total":sum(len(x) for x in surveys.values()),"nlp_domains":len(docs_by),
        "ml_tasks":len(configs),"rosstat_years":int(ROSSTAT_SCHEMA.year.nunique()) if len(ROSSTAT_SCHEMA) else 0,
        "runtime_sec":round(time.perf_counter()-t0,1)}])

    with pd.ExcelWriter(RESULTS/"CHAPTER4_FINAL_ANALYSIS.xlsx",engine="xlsxwriter") as w:
        tables={
            "load_audit":LOAD_AUDIT,"microdata_index":MICRODATA_INDEX,"survey_audit":SURVEY_AUDIT,
            "weighted_distributions":WEIGHTED_DISTRIBUTIONS,
            "nlp_fine_search":NLP_FINE_SEARCH,"nlp_fine_clusters":NLP_FINE_SUMMARY,"nlp_fine_stability":NLP_FINE_STABILITY,
            "nlp_macro_search":NLP_MACRO_SEARCH,"nlp_macro_clusters":NLP_MACRO_SUMMARY,"nlp_macro_stability":NLP_MACRO_STABILITY,
            "nlp_macro_enrichment":NLP_MACRO_ENRICHMENT,"nlp_macro_prototypes":NLP_MACRO_PROTOTYPES,
            "nlp_associations":NLP_ASSOCIATIONS,"semantic_alignment":SEMANTIC_ALIGNMENT,"semantic_null":SEMANTIC_NULL,
            "ml_task_audit":ML_TASK_AUDIT,"ml_cv_folds":ML_FOLD,"ml_metrics":ML_OVERALL,"ml_bootstrap_ci":ML_CI,
            "ml_probability_selection":ML_PROBABILITY_SELECTION,"ml_calibration":ML_CALIBRATION,
            "ml_feature_stability":ML_FEATURE_STABILITY,"ml_shap_global":ML_SHAP_GLOBAL,"ml_shap_class":ML_SHAP_CLASS,
            "ml_observed_profiles":ML_GROUP_PROFILES,"ml_associations":ML_ASSOCIATIONS,
            "rosstat_key_trends":ROSSTAT_KEY_TRENDS,"rosstat_schema":ROSSTAT_SCHEMA,
            "figure_index":FIGURE_INDEX,"experiment_audit":EXPERIMENT_AUDIT
        }
        for sh,df in tables.items():df.to_excel(w,sh,index=False)
        rap=LOADER/"ROSSTAT_YEAR_AUDIT.xlsx"
        if rap.exists():pd.read_excel(rap).to_excel(w,"rosstat_year_audit",index=False)

    # Machine-readable core
    NLP_MACRO_SUMMARY.to_csv(RESULTS/"nlp_macro_clusters.csv",index=False,encoding="utf-8-sig")
    SEMANTIC_ALIGNMENT.to_csv(RESULTS/"semantic_alignment.csv",index=False,encoding="utf-8-sig")
    ML_OVERALL.to_csv(RESULTS/"ml_metrics.csv",index=False,encoding="utf-8-sig")
    ML_CI.to_csv(RESULTS/"ml_bootstrap_ci.csv",index=False,encoding="utf-8-sig")
    ROSSTAT_KEY_TRENDS.to_csv(RESULTS/"rosstat_key_trends.csv",index=False,encoding="utf-8-sig")
    pb.update(1)

print("ФИНАЛЬНЫЙ ЭКСПЕРИМЕНТ ЗАВЕРШЁН")
display(EXPERIMENT_AUDIT.T.rename(columns={0:"value"}))
print("\nNLP макротипы:");display(NLP_MACRO_SUMMARY)
print("\nML:");display(ML_OVERALL)
print("\nProbability model selection:");display(ML_PROBABILITY_SELECTION)
print("\nSemantic alignment:");display(SEMANTIC_ALIGNMENT)


FINAL CHAPTER IV:   0%|          | 0/12 [00:00<?] 

NLP domains:   0%|          | 0/4 [00:00<?, ?it/s]

ML tasks:   0%|          | 0/4 [00:00<?, ?it/s]

ФИНАЛЬНЫЙ ЭКСПЕРИМЕНТ ЗАВЕРШЁН


,value
status,OK
encoder,intfloat/multilingual-e5-large
surveys_loaded,3
respondents_total,4809
nlp_domains,4
ml_tasks,4
rosstat_years,4
runtime_sec,1097.4


...

,task,model,macro_f1,balanced_accuracy,log_loss,brier
0,ii1_core,WeightedPrior,0.276423,0.333333,0.784665,0.449795
1,ii1_core,SplineLogitBalanced,0.381343,0.461559,1.048942,0.648069
2,ii1_core,CatBoostBalanced,0.412959,0.465324,0.957750,0.591689
3,ii1_core,SplineLogitSurvey,0.313108,0.343609,0.761228,0.440832
4,ii1_core,CatBoostSurvey,0.281047,0.334233,0.749020,0.436030
5,ii1_extended,WeightedPrior,0.276423,0.333333,0.784665,0.449795
6,ii1_extended,SplineLogitBalanced,0.407284,0.471008,1.044126,0.640244
7,ii1_extended,CatBoostBalanced,0.434963,0.487575,0.923062,0.571076
8,ii1_extended,SplineLogitSurvey,0.337890,0.353122,0.778660,0.449115
9,ii1_extended,CatBoostSurvey,0.290913,0.338171,0.732703,0.429535



Probability model selection:


,task,probability_model,log_loss,brier
0,ii1_core,CatBoostSurvey,0.749020,0.436030
1,ii1_extended,CatBoostSurvey,0.732703,0.429535
2,ii4_extended,CatBoostSurvey,0.453022,0.282478
3,ii5_extended,CatBoostSurvey,0.485582,0.256525



Semantic alignment:


,n,weighted_mean_similarity,bootstrap_ci_low,bootstrap_ci_high,random_mean,random_sd,delta_vs_random,standardized_delta,permutation_p,pair,effect_interpretation
0,1098,0.893318,0.891358,0.894918,0.890013,0.000572,0.003305,5.780127,0.002494,values_vs_family,очень малый
1,1327,0.891699,0.889593,0.893713,0.887328,0.000650,0.004370,6.723405,0.002494,homeland_vs_russia,очень малый


# Финальная сборка

На выходе создаются:

- `CHAPTER4_FINAL_ANALYSIS.xlsx`;
- `INTERPRETATION_FINAL.md`;
- каталог `publication_figures` — только графики, пригодные для основного текста;
- каталог `diagnostics` — UMAP, calibration и дополнительные матрицы;
- модели CatBoost;
- `CHAPTER4_FINAL_RESEARCH.zip` — компактный архив для научной интерпретации;
- `CHAPTER4_FINAL_FULL.zip` — полный воспроизводимый архив.

In [ ]:

# D.1 Финальная сборка архивов

if not (RESULTS/"CHAPTER4_FINAL_ANALYSIS.xlsx").exists():
    raise RuntimeError("Финальный эксперимент не завершён.")

FINAL=ROOT/"FINAL";FINAL.mkdir(parents=True,exist_ok=True)
manifest=[]
for p in ROOT.rglob("*"):
    if p.is_file() and FINAL not in p.parents:
        manifest.append({"relative_path":str(p.relative_to(ROOT)),"size_bytes":p.stat().st_size,
                         "suffix":p.suffix.lower(),"sha256":sha256(p)})
MANIFEST=pd.DataFrame(manifest)
MANIFEST.to_csv(FINAL/"manifest.csv",index=False,encoding="utf-8-sig")

full_zip=shutil.make_archive("/content/CHAPTER4_FINAL_FULL","zip",root_dir=ROOT)

research=Path("/content/CHAPTER4_FINAL_RESEARCH")
if research.exists():shutil.rmtree(research)
research.mkdir(parents=True)
for name in ["CHAPTER4_FINAL_ANALYSIS.xlsx","INTERPRETATION_FINAL.md","nlp_macro_clusters.csv",
             "semantic_alignment.csv","ml_metrics.csv","ml_bootstrap_ci.csv","rosstat_key_trends.csv"]:
    p=RESULTS/name
    if p.exists():shutil.copy2(p,research/name)
if PUB.exists():shutil.copytree(PUB,research/"publication_figures",dirs_exist_ok=True)
if DIAG.exists():shutil.copytree(DIAG,research/"diagnostics",dirs_exist_ok=True)
if MODELS.exists():shutil.copytree(MODELS,research/"models",dirs_exist_ok=True)
for p in [LOADER/"LOAD_AUDIT.xlsx",LOADER/"MICRODATA_INDEX.xlsx",LOADER/"ROSSTAT_YEAR_AUDIT.xlsx",LOADER/"ROSSTAT_VARIABLE_DICTIONARY.xlsx"]:
    if p.exists():
        d=research/"loader"/p.name;d.parent.mkdir(parents=True,exist_ok=True);shutil.copy2(p,d)
shutil.copy2(FINAL/"manifest.csv",research/"manifest.csv")
research_zip=shutil.make_archive("/content/CHAPTER4_FINAL_RESEARCH","zip",root_dir=research)

print("FULL:",full_zip)
print("RESEARCH:",research_zip)


FULL: /content/CHAPTER4_FINAL_FULL.zip
RESEARCH: /content/CHAPTER4_FINAL_RESEARCH.zip


In [ ]:

# D.2 Скачать исследовательский архив, Excel и интерпретацию
try:
    from google.colab import files
    files.download("/content/CHAPTER4_FINAL_RESEARCH.zip")
    files.download(str(RESULTS/"CHAPTER4_FINAL_ANALYSIS.xlsx"))
    files.download(str(RESULTS/"INTERPRETATION_FINAL.md"))
except Exception:
    print("/content/CHAPTER4_FINAL_RESEARCH.zip")
    print(RESULTS/"CHAPTER4_FINAL_ANALYSIS.xlsx")
    print(RESULTS/"INTERPRETATION_FINAL.md")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Критерии успешного финального прогона

В конце должны выполняться одновременно:

- ВЦИОМ: 3/3 массива;
- Росстат: `microdata_ok=True`, до 4 лет;
- NLP: 4 домена, без `ID` и технических полей;
- для каждого домена сохранены fine clusters и 3–5 macro clusters;
- `nlp_macro_stability` содержит silhouette и ARI;
- `semantic_alignment` содержит `delta_vs_random` и его содержательную классификацию;
- ML разделён на balanced discrimination и survey-weighted probability;
- class-balanced probabilities не используются как оценки долей;
- SHAP считается без index mismatch;
- `ii5` автоматически помечается как ограниченный при малом классе;
- основные рисунки находятся только в `publication_figures`;
- UMAP/calibration/association matrices вынесены в `diagnostics`;
- сформированы `CHAPTER4_FINAL_ANALYSIS.xlsx` и `CHAPTER4_FINAL_RESEARCH.zip`.